In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode,col,expr,when
from pyspark.sql.types import ArrayType, IntegerType, ShortType

In [2]:
spark = SparkSession.builder.appName('python').config("spark.driver.memory", "4g").getOrCreate()
spark

25/05/07 14:05:21 WARN Utils: Your hostname, namunaacharya resolves to a loopback address: 127.0.1.1; using 10.10.42.118 instead (on interface enp2s0)
25/05/07 14:05:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/07 14:05:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
rate_file = spark.read.option('multiline','True').json('../files/in_network_file.json')

In [9]:

provider_file = spark.read.option('multiline','True').json('../files/provider_file.json')

In [6]:
rate_file.printSchema()

root
 |-- billing_code: string (nullable = true)
 |-- billing_code_type: string (nullable = true)
 |-- billing_code_type_version: string (nullable = true)
 |-- description: string (nullable = true)
 |-- name: string (nullable = true)
 |-- negotiated_rates: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- negotiated_prices: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- billing_class: string (nullable = true)
 |    |    |    |    |-- expiration_date: string (nullable = true)
 |    |    |    |    |-- negotiated_rate: double (nullable = true)
 |    |    |    |    |-- negotiated_type: string (nullable = true)
 |    |    |    |    |-- service_code: array (nullable = true)
 |    |    |    |    |    |-- element: string (containsNull = true)
 |    |    |-- provider_references: array (nullable = true)
 |    |    |    |-- element: long (containsNull = true)
 |-- negotiation_arrangement: string (null

In [10]:
provider_file.printSchema()

root
 |-- provider_group_id: long (nullable = true)
 |-- provider_groups: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- npi: array (nullable = true)
 |    |    |    |-- element: long (containsNull = true)
 |    |    |-- tin: struct (nullable = true)
 |    |    |    |-- type: string (nullable = true)
 |    |    |    |-- value: string (nullable = true)



In [11]:
provider_df = provider_file.withColumn("provider", explode("provider_groups"))
provider_npi = provider_df.withColumn("provider_npi", explode("provider.npi"))

provider_flat = provider_npi.select(
    "provider_group_id",
    col("provider_npi").alias("npi"),
    col("provider.tin.type").alias("tin_type"),
    col("provider.tin.value").alias("tin")
)

provider_flat.show(truncate=False)

+-----------------+----------+--------+----------+
|provider_group_id|npi       |tin_type|tin       |
+-----------------+----------+--------+----------+
|10001001         |1235233008|ein     |04-3267217|
|10001001         |1316041189|ein     |04-3267217|
|10001001         |1780788554|ein     |04-3267217|
|10001001         |1891068409|ein     |04-3267217|
|10001001         |1366459570|ein     |11-1562701|
|10001001         |1417915653|ein     |11-3358535|
|10001001         |1417915653|ein     |13-3888838|
|10002001         |1609829761|ein     |00-0004110|
|10002001         |1821482241|ein     |00-0004110|
|10002001         |1760986277|ein     |00-6980743|
|10002001         |1215075882|ein     |01-0550744|
|10002001         |1013917665|ein     |01-0555304|
|10002001         |1679780811|ein     |01-0555483|
|10002001         |1700093952|ein     |01-0555483|
|10002001         |1780072447|ein     |01-0555483|
|10002001         |1952532970|ein     |01-0555483|
|10002001         |1376647511|e